# 00 Setup check

PLAN.md step 0: confirm tracking and events load for all 3 Metrica games. All logic lives in `src/passmodel/load.py`; this notebook only calls it and shows the results.

Run `uv run python scripts/download_data.py` first.

In [ ]:
import pandas as pd
from kloppy import metrica

from passmodel.load import MATCH_IDS, event_type_table, game_dir, load_tracking, tracking_summary

pd.set_option("display.max_rows", 200)

## Tracking: frames, frame rate, pitch size
Loading all three takes a minute or two.

In [ ]:
tracking = {m: load_tracking(m) for m in MATCH_IDS}
pd.DataFrame({m: tracking_summary(ds) for m, ds in tracking.items()}).T.rename_axis("match_id")

## First rows of each game (wide format: one row per frame, x/y columns per player)
Coordinates are still Metrica's normalised 0-1 units here. Note the ball is NaN at the start of games 2 and 3.

In [ ]:
for m, ds in tracking.items():
    df = ds.to_df()
    print(f"Game {m}: wide df shape {df.shape}")
    display(df.iloc[:5, :10])

## Events
Game 3 also loads through kloppy (3,594 events vs 3,620 raw, a gap to check in step 1). The audit table below uses raw Metrica types so all three games share one vocabulary.

In [ ]:
ev3 = metrica.load_event(
    event_data=game_dir(3) / "Sample_Game_3_events.json",
    meta_data=game_dir(3) / "Sample_Game_3_metadata.xml",
)
len(ev3)

## Event Type x Subtype counts per game (input to decision D1)

In [ ]:
event_type_table()